In [1]:
import json
import os

from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from groq import Groq

In [2]:
PROJECT_ROOT = Path.cwd().parent

REVIEWS_PATH = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "reviewed_papers.csv"
)

QUERY_CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_query.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUT_DIR / "final_report.json"

MODEL = "llama-3.1-8b-instant"

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
df_reviews = pd.read_csv(REVIEWS_PATH)

In [5]:
with open(
    QUERY_CONFIG_PATH,
    encoding="utf-8",
) as file:
    query_config = json.load(file)

research_topic = query_config["topic"]
arxiv_query = query_config["query"]

print("Research topic:")
print(research_topic)

Research topic:
i want to investigate about covid-19 vaccine validation


In [6]:
df_approved = (
    df_reviews[
        df_reviews["approved"] == True
    ]
    .sort_values(
        "corrected_relevance_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

df_approved[
    [
        "title",
        "corrected_relevance_score",
        "url",
    ]
]

,title,corrected_relevance_score,url
0,"Debiasing hazard-based, time-varying vaccine e...",9,http://arxiv.org/abs/2511.15099v1
1,Efficient estimation of cumulative incidence c...,9,http://arxiv.org/abs/2604.13265v1
2,A simple and powerful test of vaccine waning,9,http://arxiv.org/abs/2511.21836v2
3,Nonparametric bounds for vaccine effects in ra...,9,http://arxiv.org/abs/2510.25296v2
4,Sequentially Doubly Robust Estimation of Condi...,8,http://arxiv.org/abs/2510.10372v4


In [7]:
papers_for_editor = []

for _, paper in df_approved.iterrows():

    papers_for_editor.append(
        {
            "title": paper["title"],
            "authors": paper["authors"],
            "published": str(paper["published"]),
            "url": paper["url"],
            "summary": paper["analyst_summary"],
            "main_problem": paper["main_problem"],
            "main_contribution": paper["main_contribution"],
            "applications": paper["applications"],
            "limitations": paper["limitations"],
            "relevance_score": int(
                paper["corrected_relevance_score"]
            ),
            "relevance_reason": paper[
                "final_relevance_reason"
            ],
        }
    )

In [8]:
papers_text = json.dumps(
    papers_for_editor,
    indent=2,
    ensure_ascii=False,
)

In [9]:
prompt = f"""
You are the Editor Agent in an academic research system.

The user's research topic is:

{research_topic}

You receive a list of papers that have already been analyzed and reviewed.

Your task is to create a concise research briefing.

Reviewed papers:

{papers_text}

Return only a valid JSON object with these fields:

{{
  "executive_summary": "A concise overview of the findings",
  "main_trends": [
    "Trend 1",
    "Trend 2"
  ],
  "key_differences": [
    "Difference 1",
    "Difference 2"
  ],
  "recommended_reading_order": [
    {{
      "position": 1,
      "title": "Paper title",
      "reason": "Why this paper should be read at this position"
    }}
  ],
  "final_recommendation": "A practical recommendation for the user"
}}

Rules:

- Use only the information provided.
- Do not invent facts.
- Mention common patterns across papers.
- Explain important differences between approaches.
- Recommended papers must come from the provided list.
- Keep the response concise.
- Do not use Markdown.
- Return only JSON.
"""

In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    response_format={
        "type": "json_object"
    },
    temperature=0.2,
)

raw_report = response.choices[0].message.content

print(raw_report)

{
  "executive_summary": "Recent studies have proposed various methods to improve the validation of COVID-19 vaccines, including mitigating bias in time-varying vaccine effects, estimating cumulative incidence curves, and assessing vaccine waning.",
   "main_trends": [
      "Development of new methods to improve vaccine validation",
      "Increased focus on mitigating bias and estimating cumulative incidence curves"
   ],
   "key_differences": [
      "Different approaches to mitigating bias in time-varying vaccine effects",
      "Variations in methods for estimating cumulative incidence curves"
   ],
   "recommended_reading_order": [
      {
         "position": 1,
         "title": "Debiasing hazard-based, time-varying vaccine effects using vaccine-irrelevant infections",
         "reason": "This paper proposes a method to mitigate bias in time-varying vaccine effects, making it a good starting point for understanding this topic."
      },
      {
         "position": 2,
         

In [11]:
report = json.loads(raw_report)

In [12]:
final_report = {
    "research_topic": research_topic,
    "arxiv_query": arxiv_query,
    "papers_found": len(df_reviews),
    "papers_approved": len(df_approved),
    "report": report,
    "papers": papers_for_editor,
}

In [13]:
with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_report,
        file,
        indent=4,
        ensure_ascii=False,
    )

print(f"Final report saved to:\n{REPORT_PATH}")

Final report saved to:
C:\Users\JuanOrtizAlonso\ai-paper-review-agentic\data\output\final_report.json


In [14]:
print("EXECUTIVE SUMMARY")
print(report["executive_summary"])

EXECUTIVE SUMMARY
Recent studies have proposed various methods to improve the validation of COVID-19 vaccines, including mitigating bias in time-varying vaccine effects, estimating cumulative incidence curves, and assessing vaccine waning.


In [15]:
print("KEY DIFFERENCES")

for difference in report["key_differences"]:
    print(f"- {difference}")

KEY DIFFERENCES
- Different approaches to mitigating bias in time-varying vaccine effects
- Variations in methods for estimating cumulative incidence curves


In [16]:
print("RECOMMENDED READING ORDER")

for item in report["recommended_reading_order"]:

    print(
        f'{item["position"]}. '
        f'{item["title"]}'
    )

    print(f'   {item["reason"]}')

RECOMMENDED READING ORDER
1. Debiasing hazard-based, time-varying vaccine effects using vaccine-irrelevant infections
   This paper proposes a method to mitigate bias in time-varying vaccine effects, making it a good starting point for understanding this topic.
2. Efficient estimation of cumulative incidence curves via data fusion with surrogates
   This paper proposes methods for estimating cumulative incidence curves, which is an important aspect of vaccine validation.
3. A simple and powerful test of vaccine waning
   This paper proposes a new test to assess vaccine waning, which is an important aspect of vaccine validation.


In [17]:
print("FINAL RECOMMENDATION")
print(report["final_recommendation"])

FINAL RECOMMENDATION
Read the papers in the recommended reading order to gain a comprehensive understanding of the current state of COVID-19 vaccine validation and to identify potential areas for further research.
